<a href="https://colab.research.google.com/github/PALAK7890/SectionA_Team14_JobMarketTrends/blob/main/notebooks/05_final_load_prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/DVA_Capstone'
import pandas as pd

df = pd.read_csv(
    f"{BASE_PATH}/postings_cleaned.csv",
    engine='python',
    on_bad_lines='skip'
)



In [ ]:
print("Shape:", df.shape)
print("Loaded successfully")

Shape: (120484, 29)
Loaded successfully


In [ ]:
# Cell 2 — Select final columns for Tableau export
tableau_cols = [
    'job_id',
    'company_name',
    'title',
    'job_category',
    'formatted_experience_level',
    'formatted_work_type',
    'remote_label',
    'location',
    'state',
    'normalized_salary',
    'salary_bracket',
    'pay_period',
    'views',
    'applies',
    'conversion_rate',
    'days_active',
    'application_type',
    'listing_month',
    'listing_month_name',
    'listing_year',
    'listing_dayofweek',
    'original_listed_date',
    'expiry_date'
]

df_tableau = df[tableau_cols].copy()

print("Tableau dataset shape:", df_tableau.shape)
print("\nColumn list:")
for col in df_tableau.columns:
    nulls = df_tableau[col].isnull().sum()
    dtype = df_tableau[col].dtype
    print(f"  {col:<35} dtype: {str(dtype):<10} nulls: {nulls}")

Tableau dataset shape: (120484, 23)

Column list:
  job_id                              dtype: int64      nulls: 0
  company_name                        dtype: object     nulls: 0
  title                               dtype: object     nulls: 0
  job_category                        dtype: object     nulls: 0
  formatted_experience_level          dtype: object     nulls: 0
  formatted_work_type                 dtype: object     nulls: 0
  remote_label                        dtype: object     nulls: 0
  location                            dtype: object     nulls: 0
  state                               dtype: object     nulls: 0
  normalized_salary                   dtype: float64    nulls: 85403
  salary_bracket                      dtype: object     nulls: 0
  pay_period                          dtype: object     nulls: 0
  views                               dtype: float64    nulls: 0
  applies                             dtype: float64    nulls: 97599
  conversion_rate               

In [ ]:
# Cell 3 — Final type fixes for Tableau compatibility

# Convert date columns to proper datetime strings Tableau can parse
df_tableau['original_listed_date'] = pd.to_datetime(df_tableau['original_listed_date']).dt.strftime('%Y-%m-%d')
df_tableau['expiry_date'] = pd.to_datetime(df_tableau['expiry_date']).dt.strftime('%Y-%m-%d')

# Round float columns to 2 decimal places
df_tableau['normalized_salary'] = df_tableau['normalized_salary'].round(2)
df_tableau['conversion_rate'] = df_tableau['conversion_rate'].round(2)
df_tableau['views'] = df_tableau['views'].astype('Int64')

# Define correct ordering for experience level — useful for Tableau sorting
exp_order_map = {
    'Internship': 1,
    'Entry level': 2,
    'Associate': 3,
    'Mid-Senior level': 4,
    'Director': 5,
    'Executive': 6,
    'Unspecified': 7
}
df_tableau['experience_order'] = df_tableau['formatted_experience_level'].map(exp_order_map)

# Define correct ordering for salary bracket
bracket_order_map = {
    'Under $40K': 1,
    '$40K - $60K': 2,
    '$60K - $80K': 3,
    '$80K - $100K': 4,
    '$100K - $150K': 5,
    '$150K - $200K': 6,
    '$200K+': 7,
    'Not Disclosed': 8
}
df_tableau['salary_bracket_order'] = df_tableau['salary_bracket'].map(bracket_order_map)

# Month order
month_order_map = {
    'Dec': 1, 'Jan': 2, 'Feb': 3,
    'Mar': 4, 'Apr': 5
}
df_tableau['month_order'] = df_tableau['listing_month_name'].map(month_order_map)

print("Final Tableau dataset shape:", df_tableau.shape)
print("\nNew ordering columns added:")
print(df_tableau[['formatted_experience_level', 'experience_order',
                   'salary_bracket', 'salary_bracket_order',
                   'listing_month_name', 'month_order']].drop_duplicates().sort_values('experience_order'))

Final Tableau dataset shape: (120484, 26)

New ordering columns added:
      formatted_experience_level  experience_order salary_bracket  \
1031                  Internship                 1  Not Disclosed   
1630                  Internship                 1    $60K - $80K   
699                   Internship                 1    $40K - $60K   
6396                  Internship                 1   $80K - $100K   
5215                  Internship                 1  $150K - $200K   
...                          ...               ...            ...   
17395                Unspecified                 7         $200K+   
3692                 Unspecified                 7  $150K - $200K   
5133                 Unspecified                 7  $100K - $150K   
5128                 Unspecified                 7    $40K - $60K   
7755                 Unspecified                 7   $80K - $100K   

       salary_bracket_order listing_month_name  month_order  
1031                      8           

In [ ]:
# Cell 4 — Save final Tableau-ready export
df_tableau.to_csv('postings_tableau_ready.csv', index=False)

FINAL LOAD COMPLETE — 05_final_load_prep.ipynb

File saved: postings_tableau_ready.csv
Rows      : 120,484
Columns   : 26

Column summary:
  Dimensions (categorical) : 15
  Measures (numerical)     : 11
  Date columns             : 2

Key measures for Tableau KPIs:
  Median normalized salary : $81,750
  Avg conversion rate      : 17.52%
  Total job postings       : 120,484
  Total unique companies   : 24,032
  Total unique states      : 54

Ready for Tableau Public.
Proceed to Tableau dashboard build.
